In [1]:
import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")
    print("HF token loaded")
except Exception as e:
    print("No HF token:", e)

HF token loaded


In [2]:
import os
if not os.path.isdir("/kaggle/working/fitcheck/.git"):
    !git clone https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck
%cd /kaggle/working/fitcheck
!git pull --ff-only
!git log --oneline -1

Cloning into '/kaggle/working/fitcheck'...
remote: Enumerating objects: 629, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 629 (delta 126), reused 154 (delta 63), pack-reused 403 (from 1)
Receiving objects: 100% (629/629), 1.67 MiB | 8.13 MiB/s, done.
Resolving deltas: 100% (366/366), done.
/kaggle/working/fitcheck
Already up to date.
41b488d (HEAD -> main, origin/main, origin/HEAD) fix


In [3]:
!pip install -q -e .
import os
os.environ["PYTHONPATH"] = "/kaggle/working/fitcheck"   # belt + braces for subprocesses

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done


In [4]:
!cd /tmp && python -c "from fitcheck.gpu_db import get_gpu; print('fitcheck import OK')"

fitcheck import OK


In [5]:
!python scripts/calibration_sweep.py --gpu t4 --quant none --out /kaggle/working/runs

[1/20] SmolLM2-135M-bs2-seq512-eager.json ...
    FAILED (1). Last stderr lines:
        File "/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/torchao.py", line 142, in dispatch_torchao
          if not is_torchao_available():
                 ^^^^^^^^^^^^^^^^^^^^^^
        File "/usr/local/lib/python3.12/dist-packages/peft/import_utils.py", line 143, in is_torchao_available
          raise ImportError(
      ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported
[2/20] SmolLM2-360M-bs1-seq4096-eager.json ...
    FAILED (1). Last stderr lines:
        File "/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/torchao.py", line 142, in dispatch_torchao
          if not is_torchao_available():
                 ^^^^^^^^^^^^^^^^^^^^^^
        File "/usr/local/lib/python3.12/dist-packages/peft/import_utils.py", line 143, in is_torchao_available
          raise ImportError(
      ImportError: Found an incompatibl

In [6]:
!python scripts/calibration_sweep.py --gpu t4 --quant nf4 --tag -nf4 \
    --no-repeats --out /kaggle/working/runs


usage: calibration_sweep.py [-h] --gpu GPU [--quant {none,nf4,int8}]
                            [--precision {fp16,bf16,fp32}] [--lora-r LORA_R]
                            [--out OUT] [--tag TAG] [--kernels KERNELS]
                            [--no-repeats]
calibration_sweep.py: error: argument --tag: expected one argument


In [7]:
!python -m fitcheck.calibrate /kaggle/working/runs/*.json
print("=" * 70)
!python -m fitcheck.calibrate /kaggle/working/runs/*.json --check
!cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip


calibrate: /kaggle/working/runs/*.json: cannot read ([Errno 2] No such file or directory: '/kaggle/working/runs/*.json')
calibrate: /kaggle/working/runs/*.json: cannot read ([Errno 2] No such file or directory: '/kaggle/working/runs/*.json')
-rw-r--r-- 1 root root 160 Sep 14 12:08 runs.zip
